In [ ]:
# Cell — Load and compare all sequential rounds
import json, os

BASE_DIR = "/u/vz8/unlearning/open-unlearning"
os.chdir(BASE_DIR)

rounds = {
    "Round 0 (base)":     "saves/eval/sequential_round0_base",
    "Round 1 (forget01)": "saves/eval/sequential_round1_eval",
    "Round 2 (forget05)": "saves/eval/sequential_round2_eval",
    "Round 3 (forget10)": "saves/eval/sequential_round3_eval",
    "After relearning":   "saves/eval/sequential_relearn_eval",
}

KEY_METRICS = ["forget_quality", "model_utility", "privleak"]

results = {}
for label, path in rounds.items():
    json_path = f"{path}/TOFU_EVAL.json"
    if os.path.exists(json_path):
        with open(json_path) as f:
            data = json.load(f)
        results[label] = {
            k: round(data[k]["agg_value"], 4)
            for k in KEY_METRICS if k in data
        }
    else:
        print(f"Not yet available: {json_path}")

# Print comparison table
print(f"{'Round':<25} {'forget_quality':>15} {'model_utility':>15} {'privleak':>12}")
print("-" * 70)
for label, metrics in results.items():
    fq = metrics.get("forget_quality", "N/A")
    mu = metrics.get("model_utility", "N/A")
    pl = metrics.get("privleak", "N/A")
    print(f"{label:<25} {str(fq):>15} {str(mu):>15} {str(pl):>12}")

In [ ]:
# Cell — Plot the degradation across rounds
import matplotlib.pyplot as plt

rounds_labels = list(results.keys())
forget_quality = [results[r].get("forget_quality", 0) for r in rounds_labels]
model_utility  = [results[r].get("model_utility", 0) for r in rounds_labels]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(forget_quality, marker='o', color='red', linewidth=2)
ax1.set_xticks(range(len(rounds_labels)))
ax1.set_xticklabels(rounds_labels, rotation=20, ha='right')
ax1.set_title("Forget Quality Across Rounds")
ax1.set_ylabel("Forget Quality (higher = better forgetting)")
ax1.grid(True, alpha=0.3)

ax2.plot(model_utility, marker='o', color='blue', linewidth=2)
ax2.set_xticks(range(len(rounds_labels)))
ax2.set_xticklabels(rounds_labels, rotation=20, ha='right')
ax2.set_title("Model Utility Across Rounds")
ax2.set_ylabel("Model Utility (higher = better retention)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("sequential_unlearning_results.png", dpi=150)
plt.show()
print("Saved to sequential_unlearning_results.png")

In [ ]:
# Cell — Before vs after relearning attack on forget set
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch

forget10 = load_dataset("locuslab/TOFU", "forget10")["train"]
tokenizer = AutoTokenizer.from_pretrained(
    "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"
)

def ask(model_path, question, max_new_tokens=80):
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.float16, device_map="auto"
    )
    inputs = tokenizer(
        f"Question: {question}\nAnswer:",
        return_tensors="pt"
    ).to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    del mdl
    torch.cuda.empty_cache()
    return tokenizer.decode(out[0], skip_special_tokens=True).split("Answer:")[-1].strip()

models = {
    "After round3 unlearning": f"{BASE_DIR}/saves/models/sequential_round3",
    "After relearning attack":  f"{BASE_DIR}/saves/models/sequential_relearn",
}

print("=== Relearning Attack — Does forgotten knowledge come back? ===\n")
for i in range(3):
    q = forget10[i]["question"]
    expected = forget10[i]["answer"]
    print(f"Q: {q}")
    print(f"  Expected : {expected}")
    for label, path in models.items():
        if os.path.exists(path):
            print(f"  {label}: {ask(path, q)}")
    print()